# HUD Housing Data - Exploratory Analysis

**Data Sources:**
- `hud_chas_places`: 4 rows - Comprehensive Housing Affordability Strategy (CHAS) data
- `hud_fair_mkt_rent`: 11 rows - Fair Market Rents (FMR) by bedroom count
- `hud_safmr`: 21 rows - Small Area Fair Market Rents (SAFMR) by zip code

**Objectives:**
1. Understand HUD fair market rent standards across NJ
2. Compare metro-level FMR vs zip-level SAFMR
3. Analyze rent variations by bedroom count
4. Identify housing affordability benchmarks
5. Explore CHAS housing burden data (if available)

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to database
db_path = Path('../data/db/nj_pipeline.duckdb')
conn = duckdb.connect(str(db_path), read_only=True)
print(f"Connected to: {db_path}")

## 1. Data Overview

In [ ]:
# Schema information
print("=" * 60)
print("HUD CHAS PLACES SCHEMA")
print("=" * 60)
display(conn.execute("DESCRIBE hud_chas_places").df())

print("\n" + "=" * 60)
print("HUD FAIR MARKET RENT SCHEMA")
print("=" * 60)
display(conn.execute("DESCRIBE hud_fair_mkt_rent").df())

print("\n" + "=" * 60)
print("HUD SAFMR SCHEMA")
print("=" * 60)
display(conn.execute("DESCRIBE hud_safmr").df())

In [ ]:
# Sample data
print("\n" + "=" * 60)
print("HUD CHAS DATA")
print("=" * 60)
display(conn.execute("SELECT * FROM hud_chas_places").df())

print("\n" + "=" * 60)
print("FAIR MARKET RENT DATA")
print("=" * 60)
display(conn.execute("SELECT * FROM hud_fair_mkt_rent ORDER BY fmr_2br DESC").df())

print("\n" + "=" * 60)
print("SMALL AREA FMR DATA (Sample)")
print("=" * 60)
display(conn.execute("SELECT * FROM hud_safmr ORDER BY safmr_2br DESC LIMIT 10").df())

## 2. Fair Market Rent Analysis (Metro-Level)

In [ ]:
# Get FMR data
fmr_data = conn.execute("""
    SELECT * FROM hud_fair_mkt_rent
    ORDER BY fmr_2br DESC
""").df()

print(f"\nFair Market Rent Coverage: {len(fmr_data)} metro areas")

# Check which bedroom columns are available
bedroom_cols = [col for col in fmr_data.columns if col.startswith('fmr_')]
print(f"\nBedroom size categories available: {len(bedroom_cols)}")
for col in bedroom_cols:
    print(f"  • {col}")

In [ ]:
# Summary statistics
if bedroom_cols:
    print("\nFair Market Rent Summary Statistics:")
    display(fmr_data[bedroom_cols].describe().round(0))

In [ ]:
# Visualize FMR by bedroom count for top metro areas
if bedroom_cols and 'metro_name' in fmr_data.columns:
    # Prepare data for plotting
    fmr_plot_data = fmr_data.copy()
    fmr_plot_data = fmr_plot_data.sort_values('fmr_2br', ascending=True)
    
    fig, ax = plt.subplots(figsize=(14, 10))
    
    x = np.arange(len(fmr_plot_data))
    width = 0.15
    
    # Plot up to 5 bedroom sizes
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, col in enumerate(bedroom_cols[:5]):
        offset = (i - 2) * width
        label = col.replace('fmr_', '').replace('br', ' BR').replace('0', 'Studio')
        ax.barh(x + offset, fmr_plot_data[col], width, label=label, alpha=0.8, color=colors[i])
    
    ax.set_yticks(x)
    ax.set_yticklabels(fmr_plot_data['metro_name'], fontsize=9)
    ax.set_xlabel('Fair Market Rent ($)')
    ax.set_title('HUD Fair Market Rents by Metro Area and Bedroom Count')
    ax.legend(title='Unit Size', loc='lower right')
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare rent increase by bedroom count
if 'fmr_2br' in fmr_data.columns and 'fmr_1br' in fmr_data.columns:
    fmr_data['pct_increase_1br_to_2br'] = ((fmr_data['fmr_2br'] - fmr_data['fmr_1br']) / 
                                            fmr_data['fmr_1br'] * 100).round(1)
    
    if 'fmr_3br' in fmr_data.columns:
        fmr_data['pct_increase_2br_to_3br'] = ((fmr_data['fmr_3br'] - fmr_data['fmr_2br']) / 
                                                fmr_data['fmr_2br'] * 100).round(1)
    
    print("\nRent Increase Analysis:")
    if 'metro_name' in fmr_data.columns:
        display_cols = ['metro_name', 'fmr_1br', 'fmr_2br', 'pct_increase_1br_to_2br']
        if 'fmr_3br' in fmr_data.columns:
            display_cols.extend(['fmr_3br', 'pct_increase_2br_to_3br'])
        display(fmr_data[display_cols])

## 3. Small Area Fair Market Rent (SAFMR) Analysis

In [ ]:
# Get SAFMR data
safmr_data = conn.execute("""
    SELECT * FROM hud_safmr
    ORDER BY safmr_2br DESC
""").df()

print(f"\nSmall Area FMR Coverage: {len(safmr_data)} zip codes")

# Check which bedroom columns are available
safmr_bedroom_cols = [col for col in safmr_data.columns if col.startswith('safmr_')]
print(f"\nBedroom size categories available: {len(safmr_bedroom_cols)}")

In [ ]:
# Summary statistics
if safmr_bedroom_cols:
    print("\nSmall Area FMR Summary Statistics:")
    display(safmr_data[safmr_bedroom_cols].describe().round(0))

In [ ]:
# Top and bottom zip codes by 2BR SAFMR
if 'safmr_2br' in safmr_data.columns and 'zip_code' in safmr_data.columns:
    print("\nTop 10 Most Expensive Zip Codes (by 2BR SAFMR):")
    display(safmr_data.nlargest(10, 'safmr_2br')[['zip_code'] + safmr_bedroom_cols])
    
    print("\nBottom 10 Least Expensive Zip Codes (by 2BR SAFMR):")
    display(safmr_data.nsmallest(10, 'safmr_2br')[['zip_code'] + safmr_bedroom_cols])

In [ ]:
# Distribution of 2BR SAFMR
if 'safmr_2br' in safmr_data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(safmr_data['safmr_2br'].dropna(), bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(safmr_data['safmr_2br'].median(), color='red', linestyle='--', 
                   label=f'Median: ${safmr_data["safmr_2br"].median():,.0f}')
    axes[0].set_xlabel('2BR SAFMR ($)')
    axes[0].set_ylabel('Number of Zip Codes')
    axes[0].set_title('Distribution of 2BR Small Area FMR')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot for all bedroom sizes
    if len(safmr_bedroom_cols) > 1:
        safmr_data[safmr_bedroom_cols].boxplot(ax=axes[1])
        axes[1].set_ylabel('SAFMR ($)')
        axes[1].set_title('SAFMR Distribution by Bedroom Count')
        axes[1].set_xticklabels([col.replace('safmr_', '').replace('br', ' BR').replace('0', 'Studio') 
                                 for col in safmr_bedroom_cols], rotation=45)
        axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. FMR vs SAFMR Comparison

In [ ]:
# Compare metro-level FMR to zip-level SAFMR ranges
if 'safmr_2br' in safmr_data.columns and 'fmr_2br' in fmr_data.columns:
    print("\nComparison: Metro FMR vs Zip-Level SAFMR (2BR units)")
    print("\nMetro-Level FMR:")
    if 'metro_name' in fmr_data.columns:
        for _, row in fmr_data.iterrows():
            print(f"  {row['metro_name']}: ${row['fmr_2br']:,.0f}")
    
    print("\nZip-Level SAFMR Range:")
    print(f"  Minimum: ${safmr_data['safmr_2br'].min():,.0f}")
    print(f"  Median:  ${safmr_data['safmr_2br'].median():,.0f}")
    print(f"  Maximum: ${safmr_data['safmr_2br'].max():,.0f}")
    print(f"  Range:   ${safmr_data['safmr_2br'].max() - safmr_data['safmr_2br'].min():,.0f}")
    
    print("\nSAFMR allows for more granular, zip-code level rent standards")
    print("compared to the broader metro-area FMR approach.")

## 5. Affordability Benchmarks

In [ ]:
# Calculate income needed to afford FMR (30% rule)
# Households should spend no more than 30% of income on rent

if 'safmr_2br' in safmr_data.columns:
    safmr_data['income_needed_30pct'] = (safmr_data['safmr_2br'] * 12 / 0.30).round(0)
    safmr_data['hourly_wage_needed'] = (safmr_data['income_needed_30pct'] / 2080).round(2)  # 40hrs/wk * 52wks
    
    print("\nAffordability Analysis (2BR units, 30% income rule):")
    print("\nIncome needed to afford SAFMR:")
    print(f"  Minimum:  ${safmr_data['income_needed_30pct'].min():,.0f}/year (${safmr_data['hourly_wage_needed'].min():.2f}/hour)")
    print(f"  Median:   ${safmr_data['income_needed_30pct'].median():,.0f}/year (${safmr_data['hourly_wage_needed'].median():.2f}/hour)")
    print(f"  Maximum:  ${safmr_data['income_needed_30pct'].max():,.0f}/year (${safmr_data['hourly_wage_needed'].max():.2f}/hour)")
    
    # Compare to NJ minimum wage (example: $15/hour)
    min_wage = 15.00
    min_wage_annual = min_wage * 2080
    affordable_at_min_wage = safmr_data[safmr_data['income_needed_30pct'] <= min_wage_annual]
    
    print(f"\nAt NJ minimum wage (${min_wage}/hour = ${min_wage_annual:,.0f}/year):")
    print(f"  Affordable zip codes: {len(affordable_at_min_wage)} out of {len(safmr_data)} ({len(affordable_at_min_wage)/len(safmr_data)*100:.1f}%)")

In [ ]:
# Visualize income needed
if 'income_needed_30pct' in safmr_data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Annual income needed
    axes[0].hist(safmr_data['income_needed_30pct'].dropna() / 1000, bins=20, 
                edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].set_xlabel('Annual Income Needed ($1000s)')
    axes[0].set_ylabel('Number of Zip Codes')
    axes[0].set_title('Income Needed to Afford 2BR SAFMR\n(30% of income rule)')
    axes[0].axvline(safmr_data['income_needed_30pct'].median() / 1000, 
                   color='red', linestyle='--', 
                   label=f'Median: ${safmr_data["income_needed_30pct"].median()/1000:.0f}k')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Hourly wage needed
    axes[1].hist(safmr_data['hourly_wage_needed'].dropna(), bins=20, 
                edgecolor='black', alpha=0.7, color='darkorange')
    axes[1].set_xlabel('Hourly Wage Needed ($)')
    axes[1].set_ylabel('Number of Zip Codes')
    axes[1].set_title('Hourly Wage Needed to Afford 2BR SAFMR\n(40 hrs/week, 52 weeks/year)')
    axes[1].axvline(safmr_data['hourly_wage_needed'].median(), 
                   color='red', linestyle='--', 
                   label=f'Median: ${safmr_data["hourly_wage_needed"].median():.2f}/hr')
    axes[1].axvline(min_wage, color='green', linestyle=':', 
                   label=f'NJ Min Wage: ${min_wage}/hr', alpha=0.7)
    axes[1].legend()
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. CHAS Data Analysis

In [ ]:
# Analyze CHAS data if available
chas_data = conn.execute("SELECT * FROM hud_chas_places").df()

if len(chas_data) > 0:
    print("\nCHAS (Comprehensive Housing Affordability Strategy) Data:")
    print(f"Records: {len(chas_data)}")
    print("\nColumns available:")
    for col in chas_data.columns:
        print(f"  • {col}")
    
    print("\nSample Data:")
    display(chas_data.head(10))
else:
    print("\nCHAS data available but limited. May need to re-run pipeline with full CHAS extraction.")

## 7. Geographic Rent Patterns

In [ ]:
# Analyze SAFMR by zip code prefix (to identify regional patterns)
if 'zip_code' in safmr_data.columns and 'safmr_2br' in safmr_data.columns:
    safmr_data['zip_prefix'] = safmr_data['zip_code'].astype(str).str[:3]
    
    prefix_summary = safmr_data.groupby('zip_prefix').agg({
        'safmr_2br': ['count', 'mean', 'median', 'min', 'max']
    }).round(0)
    
    prefix_summary.columns = ['Count', 'Mean', 'Median', 'Min', 'Max']
    prefix_summary = prefix_summary.sort_values('Median', ascending=False)
    
    print("\n2BR SAFMR by Zip Code Prefix (Regional Patterns):")
    display(prefix_summary)
    
    # Note: 070xx = Northern NJ, 071xx = North Central, 072xx = Central Shore, 
    #       073xx = Central, 074xx = Central/Shore, 075xx = Camden area,
    #       076xx = West/Northwest, 077xx = North, 078xx = Central/East,
    #       079xx = Southwest, 080xx = South/Shore, 081xx = South, 082xx = South Shore, 
    #       083xx = Atlantic City area, 084xx = Cape May

## 8. Key Findings Summary

In [ ]:
print("="*70)
print("KEY FINDINGS - HUD HOUSING DATA")
print("="*70)

print(f"\n1. DATA COVERAGE")
print(f"   • Metro-level FMR: {len(fmr_data)} metro areas")
print(f"   • Zip-level SAFMR: {len(safmr_data)} zip codes")
print(f"   • CHAS records: {len(chas_data)}")

if 'fmr_2br' in fmr_data.columns:
    print(f"\n2. METRO-LEVEL FMR (2BR units)")
    print(f"   • Highest metro: ${fmr_data['fmr_2br'].max():,.0f}")
    print(f"   • Lowest metro:  ${fmr_data['fmr_2br'].min():,.0f}")
    print(f"   • Average:       ${fmr_data['fmr_2br'].mean():,.0f}")

if 'safmr_2br' in safmr_data.columns:
    print(f"\n3. ZIP-LEVEL SAFMR (2BR units)")
    print(f"   • Highest zip: ${safmr_data['safmr_2br'].max():,.0f}")
    print(f"   • Median zip:  ${safmr_data['safmr_2br'].median():,.0f}")
    print(f"   • Lowest zip:  ${safmr_data['safmr_2br'].min():,.0f}")
    print(f"   • Range:       ${safmr_data['safmr_2br'].max() - safmr_data['safmr_2br'].min():,.0f}")

if 'income_needed_30pct' in safmr_data.columns:
    print(f"\n4. AFFORDABILITY (2BR, 30% rule)")
    print(f"   • Median income needed:      ${safmr_data['income_needed_30pct'].median():,.0f}/year")
    print(f"   • Median hourly wage needed: ${safmr_data['hourly_wage_needed'].median():.2f}/hour")
    if len(affordable_at_min_wage) > 0:
        print(f"   • Affordable at min wage:    {len(affordable_at_min_wage)}/{len(safmr_data)} zips ({len(affordable_at_min_wage)/len(safmr_data)*100:.1f}%)")

if bedroom_cols and len(bedroom_cols) > 2:
    print(f"\n5. BEDROOM SIZE VARIATIONS")
    if 'fmr_1br' in fmr_data.columns and 'fmr_2br' in fmr_data.columns:
        avg_increase = fmr_data['pct_increase_1br_to_2br'].mean()
        print(f"   • Avg increase from 1BR to 2BR: {avg_increase:.1f}%")
    if 'fmr_2br' in fmr_data.columns and 'fmr_3br' in fmr_data.columns:
        avg_increase_3br = fmr_data['pct_increase_2br_to_3br'].mean()
        print(f"   • Avg increase from 2BR to 3BR: {avg_increase_3br:.1f}%")

print("\n" + "="*70)

In [ ]:
conn.close()
print("\nAnalysis complete. Database connection closed.")